## Cell Phone Lens System Example ##

In this example, we will do some simple analysis of Example 3 of the Cell Phone System described in US Patent 7535658.  

In [ ]:
import copy

import matplotlib.pyplot as plt
import numpy as np

from prtlab import (
    double_pole_basis_vectors,
    example_cell_phone_lens_ex3_system,
    plot_basis_vectors_on_sphere,
    plot_prt_lens_cross_section,
    plot_prt_ray_trace,
    polarization_ray_trace,
    sp_basis,
    transform_p_to_jones,
    update_clear_apertures_from_ray_trace,
)


First let's load this example, which contains several asphere lenses.

In [ ]:
T = example_cell_phone_lens_ex3_system()
T


These are all isotropic materials, but that does not mean there isn't value in doing some polarization ray tracing.  

Let's do some visualization of this, just so we know what we are dealing with.  

In [ ]:
Ein = np.array([1, 0])
k = np.array([0, np.sin(np.deg2rad(30)), np.cos(np.deg2rad(30))])
x1 = np.array([0, 0.775, 0])
x2 = np.array([0, -0.775, 0])
options = {"minRelativeFlux": 0.25}

d1 = polarization_ray_trace(T, k, x1, Ein, options)
d2 = polarization_ray_trace(T, k, x2, Ein, options)

Tplot = copy.deepcopy(T)
update_clear_apertures_from_ray_trace(
    Tplot, [d1, d2], margin=1.30, minimum=0.25
)

k2 = np.array([0, np.sin(np.deg2rad(20)), np.cos(np.deg2rad(20))])
k3 = np.array([0, 0, 1])
d3 = polarization_ray_trace(Tplot, k2, x1, Ein, options)
d4 = polarization_ray_trace(Tplot, k2, x2, Ein, options)
d5 = polarization_ray_trace(Tplot, k3, x1, Ein, options)
d6 = polarization_ray_trace(Tplot, k3, x2, Ein, options)
axis, _ = plot_prt_lens_cross_section(Tplot)
_ = plot_prt_ray_trace([d1, d2, d3, d4, d5, d6], ax=axis)
T = Tplot


You can see the fun shapes of this system, and some field curvature.  This is not a ray tracing program.  In theory you could interface with real optical design software to provide ray intercepts, positions, and surface normals, but that is not currently supported.

Note that I defined a separate Tplot above.  That is not strictly necessary, but the clear apertures used are only for plotting purposes.  The actual polarization ray tracing code does not care about clear apertures, it assumes they are always large enough to propagate the next ray.

Now let's look at what happens to the exit pupil of this system if you place it between two crossed polarizers.

In [ ]:
pupil_axis = np.linspace(-0.775, 0.775, 51)
X, Y = np.meshgrid(pupil_axis, pupil_axis)
pupil = np.hypot(X, Y) <= 0.775
E_in = np.array([1, 0])
k_in = np.array([0, 0, 1])
coordinate = {
    "type": "doublePole", "a_loc": k_in,
    "x_o": np.array([1, 0, 0]),
}
Jall = np.zeros((*X.shape, 2, 2), dtype=complex)
I_out = np.zeros_like(X)

for index in np.ndindex(X.shape):
    pos_in = np.array([X[index], Y[index], 0])
    ray_output = polarization_ray_trace(
        T, k_in, pos_in, E_in, options
    )
    if ray_output.final_ray_ids:
        final_ray = ray_output.rays[
            ray_output.final_ray_ids[0] - 1
        ]
        J = transform_p_to_jones(
            final_ray.P, k_in, final_ray.k, coordinate
        )
        Jall[index] = J
        E_out = np.array([0, 1]) @ J @ E_in
        I_out[index] = np.abs(E_out) ** 2

plt.figure()
plt.imshow(I_out * pupil)
_ = plt.colorbar()


You can see the classic Maltese Cross Pattern.  The value is quite low so it is not an issue for this lens, but this is just an example of how to use prtLab to do this analysis.

The other use of this example, as mentioned in PLAOS Chapter 11, is to show possible artifacts in the Jones pupil output due to choice of coordinate systems. 

First, let's show the local x coordinates for double pole coordinates.  

In [ ]:
tta = 7
k_in = np.array([0, np.sin(np.deg2rad(tta)), np.cos(np.deg2rad(tta))])
pupil_axis = np.linspace(
    -0.741676704231623, 0.741676704231623, 17
)
X, Y = np.meshgrid(pupil_axis, pupil_axis)
Ein = np.array([1, 0])
pupil = np.hypot(X, Y) <= 0.741676704231623
x_loc, y_loc, k_loc = [], [], []
coordinate = {
    "type": "doublePole", "a_loc": k_in,
    "x_o": np.array([1, 0, 0]),
}

for index in np.ndindex(X.shape):
    ray_output = polarization_ray_trace(
        T, k_in, [X[index], Y[index], 0], Ein, options
    )
    if not ray_output.final_ray_ids:
        continue
    final_ray = ray_output.rays[ray_output.final_ray_ids[0] - 1]
    if (
        np.max(np.abs(np.imag(final_ray.k))) < 1e-12
        and pupil[index]
    ):
        x_out, y_out = double_pole_basis_vectors(
            final_ray.k, coordinate["a_loc"], coordinate["x_o"]
        )
        x_loc.append(x_out)
        y_loc.append(y_out)
        k_loc.append(final_ray.k)

axis, _ = plot_basis_vectors_on_sphere(
    np.real(k_loc), np.real(x_loc), None,
    show_sphere=False, arrow_scale=0.02,
    line_width=1, view=(0, tta, -90),
)
axis.set_axis_off()


This snippet of code is filling the entrance pupil of the system for an angle of incidence of 7 degrees and storing the local coordinate system at the image plane based on double pole coordinates.  Because of the choice of a_loc, there is no singularity in the coordinate system showing up.

But, if one uses s/p coordinates for the local exit coordinates, a singularity will be seen at the bottom, as k_o approaches normal incidence:

In [ ]:
# This is the only difference from the last snippet.
x_loc, y_loc, k_loc = [], [], []
normal = np.array([0, 0, 1])

for index in np.ndindex(X.shape):
    ray_output = polarization_ray_trace(
        T, k_in, [X[index], Y[index], 0], Ein, options
    )
    if not ray_output.final_ray_ids:
        continue
    final_ray = ray_output.rays[ray_output.final_ray_ids[0] - 1]
    if (
        np.max(np.abs(np.imag(final_ray.k))) < 1e-12
        and pupil[index]
    ):
        p_out, s_out = sp_basis(final_ray.k, normal)
        x_loc.append(s_out)
        y_loc.append(p_out)
        k_loc.append(final_ray.k)

axis, _ = plot_basis_vectors_on_sphere(
    np.real(k_loc), np.real(x_loc), None,
    show_sphere=False, arrow_scale=0.02,
    line_width=1, view=(0, tta, -90),
)
axis.set_axis_off()


Here you can see the singularity as the output k vector approaches normal incidence.  WHile not shown, this can lead to artifacts in the Jones pupil that may appear interesting, but are really just a consequence of the local coordinate transformation.